# 04 — Analyze results

Three steps: **configure** → **results** → **inspect**.

Prerequisite: notebook 03 experiment runs under `outputs/runs/`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments.notebook_helpers import (
    AnalyzeResultsConfig,
    filter_summary,
    load_comparison,
    resolve_figures_dir,
    resolve_runs_root,
    run_analysis_plots,
    top_k_errors,
)
from src.experiments.results import load_all_runs, load_registry
from src.utils import setup_logging
from src.visualization import build_image_context, visualize_image, visualize_measurement

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
runs_root = resolve_runs_root()
print(f"Storage: {STORAGE} | runs: {runs_root}")


## 1. Configure


In [ ]:
ANALYSIS_CFG = AnalyzeResultsConfig(
    run_names=[
        "baseline_skeleton_validation_lengths2",
        "regression_skeleton_validation_lengths2",
    ],
    inspect_run="regression_skeleton_validation_lengths2",

    plot_mae_comparison=True,
    plot_rmse_comparison=True,
    plot_error_histogram=True,
    plot_pred_vs_gt=True,
    plot_residual=True,
    plot_pivot_table=False,
    plot_per_image_viz=False,   # True to debug worst images inline

    top_k_worst=5,
    top_k_best=5,
    save_viz=False,
)


## 2. Results & plots


In [ ]:
registry = load_registry(runs_root / "experiments.csv")
all_runs = load_all_runs(runs_root)
summary = filter_summary(registry if len(registry) else all_runs, ANALYSIS_CFG.run_names)

if len(summary):
    cols = [c for c in ["run_name", "pipeline", "method", "mae_mm", "rmse_mm", "n_samples"] if c in summary.columns]
    display(summary[cols].sort_values("mae_mm"))
    run_analysis_plots(summary, ANALYSIS_CFG, runs_root, show=True)
else:
    print("No runs matched — update ANALYSIS_CFG.run_names or run notebook 03.")


## 3. Inspect


In [ ]:
inspect_dir = runs_root / ANALYSIS_CFG.inspect_run
comparison = load_comparison(inspect_dir)

if comparison is None:
    print(f"No comparison.csv at {inspect_dir}")
else:
    show_cols = [c for c in ["image_id", "length_mm", "predicted_length_mm", "error_mm", "abs_error_mm"] if c in comparison.columns]
    if "skeleton_length_mm" in comparison.columns:
        show_cols.append("skeleton_length_mm")
    print(f"=== Worst {ANALYSIS_CFG.top_k_worst} — {ANALYSIS_CFG.inspect_run} ===")
    display(top_k_errors(comparison, k=ANALYSIS_CFG.top_k_worst, worst=True)[show_cols])
    print(f"\n=== Best {ANALYSIS_CFG.top_k_best} ===")
    display(top_k_errors(comparison, k=ANALYSIS_CFG.top_k_best, worst=False)[show_cols])

    if ANALYSIS_CFG.plot_per_image_viz:
        for image_id in top_k_errors(comparison, k=3, worst=True)["image_id"].astype(str):
            print(f"--- {image_id} ---")
            ctx = build_image_context(image_id, run_dir=inspect_dir)
            visualize_image(ctx, debug=True)
            visualize_measurement(ctx, debug=True)
